# MOSAIC Orchestrator on Google Colab

This notebook sets up a MOSAIC orchestrator that runs R + reticulate locally on Colab, while Coiled workers handle LASER simulations on Azure.

## Setup Instructions

### 1. Add Coiled Token as a Colab Secret

1. Go to https://cloud.coiled.io/profile and copy your **API token**
2. In this notebook, click the **key icon** (🔑) in the left sidebar → "Secrets"
3. Add a new secret:
   - **Name:** `COILED_TOKEN`
   - **Value:** *(paste your token)*
4. Toggle **"Notebook access"** ON for this notebook

### 2. (Optional) Mount Google Drive for Persistent Output

Colab wipes `/content/` on disconnect. To keep results, mount Drive and set `OUTPUT_TO_DRIVE = True` in the config cell below.

### 3. Runtime Settings

- **GPU not needed** — the orchestrator is CPU-only (R likelihood calculations)
- **High-RAM recommended** for large calibrations (Runtime → Change runtime type → High-RAM)
- Free tier: ~12h session limit. Colab Pro: ~24h

### 4. Environment Caching

**First run** (~15-20 min): installs everything from scratch, then saves a cache to Google Drive.
**Subsequent runs** (~2-3 min): restores from cache — skips all compilation.

To force a fresh install (e.g., after a MOSAIC version bump), set `FORCE_FRESH_INSTALL = True` in the config cell.

### 5. Run All Cells

---

## Configuration

In [ ]:
#@title Configuration {display-mode: "form"}

#@markdown ### Branch & Country
MOSAIC_BRANCH = "tt/azure-coiled"  #@param ["tt/azure-coiled", "main"] {allow-input: true}
ISO_CODE = "ETH"  #@param {type: "string"}

#@markdown ### Calibration Settings
N_SIMULATIONS = 10000  #@param {type: "integer"}
BATCH_SIZE = 5000  #@param {type: "integer"}
N_ITERATIONS = 3  #@param {type: "integer"}
N_WORKERS = 50  #@param {type: "integer"}

#@markdown ### GPU Acceleration
#@markdown Enable GPU-batched likelihood (requires Colab GPU runtime).
#@markdown **Runtime > Change runtime type > T4 GPU** before enabling this.
USE_GPU = False  #@param {type: "boolean"}

#@markdown ### Coiled Settings
COILED_WORKSPACE = "idm-coiled-idmad-r2"  #@param {type: "string"}
COILED_SOFTWARE = "mosaic-acr-workers"  #@param {type: "string"}
WORKER_VM_TYPE = "Standard_D4s_v6"  #@param ["Standard_D2s_v6", "Standard_D4s_v6", "Standard_D8s_v6"] {allow-input: true}
AZURE_REGION = "westus2"  #@param {type: "string"}

#@markdown ### Output & Caching
OUTPUT_TO_DRIVE = False  #@param {type: "boolean"}
FORCE_FRESH_INSTALL = False  #@param {type: "boolean"}

#@markdown Set `FORCE_FRESH_INSTALL = True` to rebuild the environment from scratch (e.g., after a MOSAIC version bump). The new environment will be re-cached automatically.

print(f"Config: {ISO_CODE}, {N_SIMULATIONS} sims, {N_WORKERS} workers on {WORKER_VM_TYPE}")
if USE_GPU:
    print("GPU: ON (torch batched likelihood)")
if FORCE_FRESH_INSTALL:
    print("FORCE_FRESH_INSTALL is ON -- will ignore cache and rebuild from scratch")

## Environment Setup

This single cell handles the entire environment:
- **Cache exists on Drive?** Restore in ~2-3 min (system deps still installed via apt, but R/Python/MOSAIC are restored from tarball).
- **No cache?** Full install (~15-20 min), then saves cache to Drive for next time.

The cache lives at `My Drive/MOSAIC/.cache/mosaic_env.tar.zst`.

In [ ]:
import os, subprocess

# Mount Google Drive (needed for both cache and output)
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/MOSAIC/.cache'
CACHE_FILE = os.path.join(CACHE_DIR, 'mosaic_env.tar.zst')
os.makedirs(CACHE_DIR, exist_ok=True)

try:
    force = FORCE_FRESH_INSTALL
except NameError:
    force = False

CACHE_HIT = os.path.exists(CACHE_FILE) and not force

if CACHE_HIT:
    print(f"✓ Cache found: {CACHE_FILE}")
    size_gb = os.path.getsize(CACHE_FILE) / (1024**3)
    print(f"  Size: {size_gb:.1f} GB — will restore instead of reinstalling")
else:
    if force:
        print("⚠ FORCE_FRESH_INSTALL — ignoring cache, will build from scratch")
    else:
        print("No cache found — will do full install and save cache after")

# Export env vars for bash cells
os.environ['CACHE_HIT'] = '1' if CACHE_HIT else '0'
os.environ['CACHE_FILE'] = CACHE_FILE
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['MOSAIC_BRANCH'] = MOSAIC_BRANCH if 'MOSAIC_BRANCH' in dir() else 'tt/azure-coiled'
os.environ['USE_GPU'] = str(USE_GPU) if 'USE_GPU' in dir() else 'False'

if os.environ['USE_GPU'] == 'True':
    print("GPU: ON — torch will be installed for batched likelihood")

### Step 1: System Dependencies (always needed — not cached)

In [2]:
%%bash
# System libraries needed by R geospatial packages + HDF5 for LASER
# These are OS packages and must be installed fresh each session
apt-get update -qq && apt-get install -y --no-install-recommends \
    software-properties-common dirmngr \
    libgdal-dev libproj-dev libgeos-dev libudunits2-dev \
    libhdf5-dev zlib1g-dev \
    libfontconfig1-dev libharfbuzz-dev libfribidi-dev \
    libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev \
    libcurl4-openssl-dev libssl-dev libxml2-dev \
    cmake gfortran zstd htop nvtop tree \
    > /dev/null 2>&1

# Install R if not present (Colab may ship with an older version)
if ! R --version 2>/dev/null | head -1 | grep -qE '4\.[4-9]'; then
    wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc | \
        gpg --dearmor -o /usr/share/keyrings/r-project.gpg
    echo "deb [signed-by=/usr/share/keyrings/r-project.gpg] https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/" \
        > /etc/apt/sources.list.d/r-project.list
    apt-get update -qq && apt-get install -y --no-install-recommends r-base r-base-dev > /dev/null 2>&1
fi

echo "✓ System deps installed, R $(R --version 2>&1 | head -1 | grep -oP 'R version \K[\d.]+')"

✓ System deps installed, R 4.5.3


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


### Step 2: Install or Restore Environment

In [3]:
%%bash
CACHE_HIT="${CACHE_HIT:-0}"
CACHE_FILE="${CACHE_FILE:-/content/drive/MyDrive/MOSAIC/.cache/mosaic_env.tar.zst}"

if [ "$CACHE_HIT" = "1" ]; then
    echo "=== Restoring environment from cache ==="
    echo "  Extracting $(du -sh "$CACHE_FILE" | cut -f1) archive..."
    cd / && tar --use-compress-program=zstd -xf "$CACHE_FILE"

    # Re-apply system library fixes (OpenSSL, libexpat)
    VENV=/root/.virtualenvs/r-mosaic
    [ -f "$VENV/lib/libcrypto.so.3" ] && cp "$VENV/lib/libcrypto.so.3" /usr/lib/x86_64-linux-gnu/libcrypto.so.3
    [ -f "$VENV/lib/libssl.so.3" ]    && cp "$VENV/lib/libssl.so.3"    /usr/lib/x86_64-linux-gnu/libssl.so.3
    [ -f "$VENV/lib/libexpat.so.1" ]  && cp "$VENV/lib/libexpat.so.1"  /usr/lib/x86_64-linux-gnu/libexpat.so.1
    ldconfig

    echo "✓ Environment restored from cache"
    R -e "cat('  MOSAIC', as.character(packageVersion('MOSAIC')), '\n')"
else
    echo "No cache — run the cells below for fresh install"
fi

No cache — run the cells below for fresh install


#### Fresh install (skip these cells if cache was restored above)

In [ ]:
%%bash
[ "$CACHE_HIT" = "1" ] && echo "✓ Skipped (restored from cache)" && exit 0

echo "[1/5] Installing R packages (~10 min)..."
R --no-save << 'RSCRIPT'
options(repos = c(CRAN = "https://cloud.r-project.org"), Ncpus = 2L)

install.packages("remotes")
cat("  ✓ remotes\n")

tryCatch(
    install.packages("arrow", repos = "https://apache.r-universe.dev"),
    error = function(e) install.packages("arrow")
)
cat("  ✓ arrow\n")

grp1 <- c("sf", "terra", "exactextractr", "hdf5r", "reticulate")
install.packages(grp1)
cat("  ✓ sf, terra, exactextractr, hdf5r, reticulate\n")

grp2 <- c("data.table", "dplyr", "ggplot2", "cowplot", "gridExtra", "viridis")
install.packages(grp2)
cat("  ✓ data.table, dplyr, ggplot2, cowplot, gridExtra, viridis\n")

grp3 <- c("elevatr", "rnaturalearth", "rnaturalearthdata", "countrycode")
install.packages(grp3)
cat("  ✓ elevatr, rnaturalearth, rnaturalearthdata, countrycode\n")

grp4 <- c("dbscan", "FNN", "furrr", "slider", "RhpcBLASctl",
           "truncnorm", "sensitivity", "jsonlite", "testthat")
install.packages(grp4)
cat("  ✓ dbscan, FNN, furrr, slider, RhpcBLASctl, truncnorm, sensitivity, jsonlite, testthat\n")

remotes::install_version("ggrepel", version = "0.9.5",
    repos = "https://cloud.r-project.org")
cat("  ✓ ggrepel\n")

# torch for GPU-batched likelihood (optional — only used when USE_GPU=TRUE)
use_gpu <- Sys.getenv("USE_GPU", "False")
if (tolower(use_gpu) == "true") {
    cat("  Installing torch (GPU enabled)...\n")
    install.packages("torch")
    torch::install_torch()
    cat("  ✓ torch (CUDA available:", torch::cuda_is_available(), ")\n")
} else {
    cat("  ⊘ Skipping torch (USE_GPU is not enabled)\n")
}

cat("[1/5] ✓ All R packages installed\n")
RSCRIPT

In [ ]:
%%bash
[ "$CACHE_HIT" = "1" ] && echo "✓ Skipped (restored from cache)" && exit 0

echo "[2/5] Installing MOSAIC R package..."
MOSAIC_BRANCH="${MOSAIC_BRANCH:-tt/azure-coiled}"
R --no-save << RSCRIPT
options(repos = c(CRAN = "https://cloud.r-project.org"))

# Retry wrapper — GitHub API can rate-limit unauthenticated Colab requests
install_ok <- FALSE
for (attempt in 1:3) {
    result <- tryCatch({
        remotes::install_github("InstituteforDiseaseModeling/MOSAIC-pkg",
            ref = "${MOSAIC_BRANCH}",
            dependencies = TRUE, upgrade = "never", force = TRUE)
        TRUE
    }, error = function(e) {
        cat("  Attempt", attempt, "failed:", conditionMessage(e), "\n")
        if (attempt < 3) {
            cat("  Retrying in", 10 * attempt, "seconds...\n")
            Sys.sleep(10 * attempt)
        }
        FALSE
    })
    if (result) { install_ok <- TRUE; break }
}
if (!install_ok) stop("Failed to install MOSAIC after 3 attempts")

library(MOSAIC)
cat("[2/5] ✓ MOSAIC", as.character(packageVersion("MOSAIC")), "installed\n")
RSCRIPT

In [6]:
%%bash
[ "$CACHE_HIT" = "1" ] && echo "✓ Skipped (restored from cache)" && exit 0

echo "[3/5] Installing Python environment (laser-cholera via conda, ~5 min)..."
R --no-save << 'RSCRIPT'
library(MOSAIC)
install_dependencies(force = TRUE)
cat("[3/5] ✓ Python environment installed\n")
RSCRIPT

[3/5] Installing Python environment (laser-cholera via conda, ~5 min)...

R version 4.5.3 (2026-03-11) -- "Reassured Reassurer"
Copyright (C) 2026 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> library(MOSAIC)
> install_dependencies(force = TRUE)
PREFIX=/root/.local/share/r-miniconda
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.2.25-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Ext


 __  __   ___   ____     _     ___  ____       __      ___    _____  _____   _____ ___
|  \/  | / _ \ / ___|   / \   |_ _|/ ___|   __/ /_    / /    /   |  / ___/ / ____// __ \
| |\/| || | | |\___ \  / _ \   | || |      /_  __/   / /    / /| |  \__ \ / __/  / /_/ /
| |  | || |_| | ___) |/ ___ \  | || |___    /_/     / /___ / ___ | ___/ // /___ / _, _/
|_|  |_| \___/ |____//_/   \_\|___|\____|          /_____//_/  |_|/____//_____//_/ |_|

Welcome to the Metapopulation Outbreak Simulation with Agent-based Implementation
for Cholera (MOSAIC) featuring the Light-agent Spatial Model for ERadication (LASER)!

Version: 0.17.33


── Installing MOSAIC Dependencies ──────────────────────────────────────────────
! No conda installation found. Installing Miniconda...
* Installing Miniconda -- please wait a moment ...
* Downloading 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh' ...
trying URL 'https://github.com/conda-forge/miniforge/releases/latest/d

In [7]:
%%bash
[ "$CACHE_HIT" = "1" ] && echo "✓ Skipped (restored from cache)" && exit 0

echo "[4/5] Adding Dask + Coiled to Python env..."
~/.virtualenvs/r-mosaic/bin/pip install --no-cache-dir dask distributed coiled 2>&1 | tail -3
~/.virtualenvs/r-mosaic/bin/pip uninstall -y tensorflow 2>/dev/null
echo "[4/5] ✓ Dask + Coiled installed, TensorFlow stripped"

# Fix native library mismatches (OpenSSL, libexpat)
VENV=/root/.virtualenvs/r-mosaic
if [ -f "$VENV/lib/libcrypto.so.3" ]; then
    cp "$VENV/lib/libcrypto.so.3" /usr/lib/x86_64-linux-gnu/libcrypto.so.3
    cp "$VENV/lib/libssl.so.3"    /usr/lib/x86_64-linux-gnu/libssl.so.3
    ldconfig
    echo "  ✓ OpenSSL fixed"
fi
if [ -f "$VENV/lib/libexpat.so.1" ]; then
    cp "$VENV/lib/libexpat.so.1" /usr/lib/x86_64-linux-gnu/libexpat.so.1
    ldconfig
    echo "  ✓ libexpat fixed"
fi

[4/5] Adding Dask + Coiled to Python env...

Found existing installation: tensorflow 2.21.0
Uninstalling tensorflow-2.21.0:
  Successfully uninstalled tensorflow-2.21.0
[4/5] ✓ Dask + Coiled installed, TensorFlow stripped
  ✓ OpenSSL fixed
  ✓ libexpat fixed


/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12

In [ ]:
%%bash
[ "$CACHE_HIT" = "1" ] && echo "✓ Skipped (restored from cache)" && exit 0

echo "[5/5] Saving environment cache to Google Drive..."
CACHE_FILE="${CACHE_FILE:-/content/drive/MyDrive/MOSAIC/.cache/mosaic_env.tar.zst}"

R_LIB=$(Rscript -e "cat(.libPaths()[1])")
echo "  R library: $R_LIB"
echo "  Conda env: /root/.virtualenvs/r-mosaic"

cd / && tar --use-compress-program="zstd -3 -T0" \
    -cf "$CACHE_FILE" \
    "$R_LIB" \
    root/.virtualenvs/r-mosaic \
    2>&1 | tail -5

echo "[5/5] ✓ Cache saved: $(du -sh "$CACHE_FILE" | cut -f1) → $CACHE_FILE"
echo "  Next session will restore in ~2-3 min instead of reinstalling"
echo ""
echo "=== Fresh install complete ==="

### Step 3: Verify Installation

In [9]:
%%bash
R --no-save << 'RSCRIPT'
library(MOSAIC)
check_dependencies()

# Verify Dask/Coiled are importable
# Note: check_dependencies() already activates the conda env via reticulate,
# so we can import directly without use_condaenv()/use_virtualenv()
library(reticulate)
dask <- import("dask.distributed")
coiled <- import("coiled")
cat("\n✓ Dask version:", dask$`__version__`, "\n")
cat("✓ Coiled importable\n")
cat("\n✓ All checks passed\n")
RSCRIPT


R version 4.5.3 (2026-03-11) -- "Reassured Reassurer"
Copyright (C) 2026 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> library(MOSAIC)
> check_dependencies()
python:         /root/.virtualenvs/r-mosaic/bin/python3.12
libpython:      /root/.virtualenvs/r-mosaic/lib/libpython3.12.so
pythonhome:     /root/.virtualenvs/r-mosaic:/root/.virtualenvs/r-mosaic
version:        3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50


 __  __   ___   ____     _     ___  ____       __      ___    _____  _____   _____ ___
|  \/  | / _ \ / ___|   / \   |_ _|/ ___|   __/ /_    / /    /   |  / ___/ / ____// __ \
| |\/| || | | |\___ \  / _ \   | || |      /_  __/   / /    / /| |  \__ \ / __/  / /_/ /
| |  | || |_| | ___) |/ ___ \  | || |___    /_/     / /___ / ___ | ___/ // /___ / _, _/
|_|  |_| \___/ |____//_/   \_\|___|\____|          /_____//_/  |_|/____//_____//_/ |_|

Welcome to the Metapopulation Outbreak Simulation with Agent-based Implementation
for Cholera (MOSAIC) featuring the Light-agent Spatial Model for ERadication (LASER)!

Version: 0.17.33


── Checking Python and R dependencies for MOSAIC ───────────────────────────────
✔ Found Python conda environment at /root/.virtualenvs/r-mosaic.
✔ Virtual Environment Root: /root/.virtualenvs/r-mosaic
✔ Python Executable: /root/.virtualenvs/r-mosaic/bin/python3.12
✔ Python Version: 3.12.13
✔ numpy: 1.26.4 (expected =1.26.4)
✔ pyarrow: 14.0.1 (expected =14.0.1)
✔ numb

### Step 4: Authenticate Coiled

This reads the `COILED_TOKEN` secret you added in the setup instructions above.

In [ ]:
import os, sys

# Read token from Colab secrets
try:
    from google.colab import userdata
    token = userdata.get('COILED_TOKEN')
    print("✓ COILED_TOKEN found in Colab secrets")
except (ImportError, userdata.SecretNotFoundError):
    token = None
    print("⚠ COILED_TOKEN not found in Colab secrets.")
    print("  Add it via the 🔑 icon in the left sidebar.")
    print("  Get your token from: https://cloud.coiled.io/profile")

if token:
    # Write Coiled config
    dask_dir = os.path.expanduser("~/.config/dask")
    os.makedirs(dask_dir, exist_ok=True)
    with open(os.path.join(dask_dir, "coiled.yaml"), "w") as f:
        f.write(f"coiled:\n  token: {token}\n")
    print("✓ Coiled credentials saved to ~/.config/dask/coiled.yaml")

    # Verify via Python API (coiled CLI 'whoami' removed in newer versions)
    sys.path.insert(0, os.path.expanduser("~/.virtualenvs/r-mosaic/lib/python3.12/site-packages"))
    try:
        import coiled
        coiled.Cloud(token=token)
        print("✓ Coiled authentication verified")
    except Exception as e:
        print(f"Warning: Could not verify Coiled auth: {e}")
        print("  The token is saved — it may still work at cluster creation time.")

### Step 5: Clone Data & Configure Output

In [ ]:
import os

# Drive is already mounted from the cache step
try:
    output_to_drive = OUTPUT_TO_DRIVE
except NameError:
    output_to_drive = False

if output_to_drive:
    output_base = '/content/drive/MyDrive/MOSAIC/output'
    os.makedirs(output_base, exist_ok=True)
    print(f"✓ Output will be saved to Google Drive: {output_base}")
else:
    output_base = '/content/output'
    os.makedirs(output_base, exist_ok=True)
    print(f"⚠ Output to local /content/output (lost on disconnect)")
    print("  Set OUTPUT_TO_DRIVE = True in config cell to persist results")

os.environ['MOSAIC_OUTPUT_BASE'] = output_base

: 

In [ ]:
%%bash
# Clone MOSAIC-data (skip if already present)
if [ -d /content/MOSAIC/MOSAIC-data ]; then
    echo "✓ MOSAIC-data already cloned"
else
    mkdir -p /content/MOSAIC
    git clone --depth 1 \
        https://github.com/InstituteforDiseaseModeling/MOSAIC-data \
        /content/MOSAIC/MOSAIC-data 2>&1 | tail -3
    echo "✓ MOSAIC-data cloned"
fi

# Create model directory structure
mkdir -p /content/MOSAIC/MOSAIC-pkg/model/{input,output}
echo "✓ Directory structure ready"

: 

---

## Run MOSAIC Calibration

Everything above this line is setup. The cell below runs the actual calibration using the config parameters from the top of the notebook.

In [ ]:
import os

# Pass Python config variables to R via environment variables
for var in ['ISO_CODE', 'N_SIMULATIONS', 'BATCH_SIZE', 'N_ITERATIONS', 'N_WORKERS',
            'COILED_WORKSPACE', 'COILED_SOFTWARE', 'WORKER_VM_TYPE', 'AZURE_REGION',
            'USE_GPU']:
    try:
        os.environ[var] = str(eval(var))
    except NameError:
        pass

gpu_status = "GPU ON" if os.environ.get('USE_GPU', 'False') == 'True' else "CPU"
print(f"Launching: {os.environ.get('ISO_CODE','ETH')} | "
      f"{os.environ.get('N_SIMULATIONS','10000')} sims | "
      f"{os.environ.get('N_WORKERS','50')} workers | "
      f"{gpu_status} | "
      f"output -> {os.environ.get('MOSAIC_OUTPUT_BASE', '/content/output')}")

In [ ]:
%%bash
R --no-save << 'RSCRIPT'
library(MOSAIC)

# ---------- Read config from environment ----------
iso_code      <- Sys.getenv("ISO_CODE", "ETH")
n_simulations <- as.integer(Sys.getenv("N_SIMULATIONS", "10000"))
batch_size    <- as.integer(Sys.getenv("BATCH_SIZE", "5000"))
n_iterations  <- as.integer(Sys.getenv("N_ITERATIONS", "3"))
n_workers     <- as.integer(Sys.getenv("N_WORKERS", "50"))
workspace     <- Sys.getenv("COILED_WORKSPACE", "idm-coiled-idmad-r2")
software      <- Sys.getenv("COILED_SOFTWARE", "mosaic-acr-workers")
vm_type       <- Sys.getenv("WORKER_VM_TYPE", "Standard_D4s_v6")
region        <- Sys.getenv("AZURE_REGION", "westus2")
output_base   <- Sys.getenv("MOSAIC_OUTPUT_BASE", "/content/output")
use_gpu       <- tolower(Sys.getenv("USE_GPU", "False")) == "true"

# ---------- Setup ----------
set_root_directory("/content/MOSAIC")

run_stamp  <- format(Sys.time(), "%Y%m%d_%H%M%S")
dir_output <- file.path(output_base, paste0(iso_code, "_", run_stamp))
cat("Output directory:", dir_output, "\n")

config <- get_location_config(iso = iso_code)
priors <- get_location_priors(iso = iso_code)

# ---------- Control ----------
ctrl <- mosaic_control_defaults()

ctrl$calibration$n_simulations <- n_simulations
ctrl$calibration$batch_size    <- batch_size
ctrl$calibration$n_iterations  <- n_iterations

ctrl$parallel$use_gpu <- use_gpu

ctrl$sampling$sample_tau_i          <- FALSE
ctrl$sampling$sample_mobility_gamma <- FALSE
ctrl$sampling$sample_mobility_omega <- FALSE

ctrl$likelihood$weight_cases  <- 1
ctrl$likelihood$weight_deaths <- 0.05

ctrl$predictions$best_model_n_sims        <- 10
ctrl$predictions$ensemble_n_sims_per_param <- 3

ctrl$npe$enable         <- FALSE
ctrl$paths$clean_output <- TRUE
ctrl$paths$plots        <- FALSE
ctrl$io <- mosaic_io_presets("fast")

# ---------- Dask/Coiled spec ----------
dask_spec <- list(
  type         = "coiled",
  n_workers    = n_workers,
  software     = software,
  workspace    = workspace,
  scheduler_vm_types = c("Standard_D8s_v6"),
  vm_types     = c(vm_type),
  timeout      = 1200,
  region       = region,
  idle_timeout = "30 minutes",
  worker_options = list(nthreads = 1L)
)

# ---------- Run ----------
cat("\n=== Starting MOSAIC calibration ===", "\n")
cat("  Country:", iso_code, "\n")
cat("  Simulations:", n_simulations, "\n")
cat("  Workers:", n_workers, "x", vm_type, "\n")
cat("  Workspace:", workspace, "\n")
cat("  GPU likelihood:", use_gpu, "\n")
if (use_gpu && requireNamespace("torch", quietly = TRUE)) {
  cat("  torch CUDA:", torch::cuda_is_available(), "\n")
}
cat("\n")

result <- run_MOSAIC(
  config     = config,
  priors     = priors,
  dir_output = dir_output,
  control    = ctrl,
  dask_spec  = dask_spec
)

cat("\n=== Done ===", "\n")
cat("Simulations:", result$summary$sims_total, "\n")
cat("Results saved to:", dir_output, "\n")
RSCRIPT

## Download Results

If you didn't use Google Drive, download the results as a zip file.

In [ ]:
import os, shutil, glob

output_base = os.environ.get('MOSAIC_OUTPUT_BASE', '/content/output')

# Find the latest output directory
output_dirs = sorted(glob.glob(os.path.join(output_base, f"{ISO_CODE}_*")))

if output_dirs:
    latest = output_dirs[-1]
    dirname = os.path.basename(latest)
    zip_path = f"/content/{dirname}"
    shutil.make_archive(zip_path, "zip", output_base, dirname)
    print(f"✓ Results zipped: {zip_path}.zip")
    print(f"  Download via the file browser (📁) in the left sidebar")

    # Auto-download in Colab
    try:
        from google.colab import files
        files.download(f"{zip_path}.zip")
    except ImportError:
        pass
else:
    print(f"No output directories found in {output_base}")

: 

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| Coiled auth fails | Re-run Step 4. Check that `COILED_TOKEN` secret is set correctly. |
| `Cannot find mosaic_dask_worker.py` | Set `FORCE_FRESH_INSTALL = True` and re-run all cells. |
| OpenSSL / libexpat errors | The cache may be stale. Set `FORCE_FRESH_INSTALL = True`. |
| Session disconnects mid-run | Use Colab Pro, reduce sims, or set `OUTPUT_TO_DRIVE = True` |
| R package compilation fails (OOM) | Use High-RAM runtime. |
| Workers stuck / timeout | Check Coiled dashboard at https://cloud.coiled.io |
| Cache too large for Drive | Free Drive has 15 GB. Cache is ~3-5 GB. Delete old output to free space. |

### Re-run Tips

- **After a reconnect:** Run all cells — Step 2 restores from cache in ~2-3 min
- **After a MOSAIC version bump:** Set `FORCE_FRESH_INSTALL = True`, run all, then set it back to `False`
- **To delete the cache:** Delete `My Drive/MOSAIC/.cache/mosaic_env.tar.zst`
- Step 4 (Coiled auth) must be re-run each session
- Step 5 (clone data) skips if already present